# Telecom Service Assurance & Cost-to-Serve Analytics Platform

### Synthetic Dataset Generation



This notebook generates the synthetic dataset used throughout this project.



The data is created using business rules based on telecom service assurance operations, including incident management, fault handling, SLA tracking, dispatch activities, and cost analysis.



After validation, the notebook exports the following datasets:



- Dim_Date.csv

- Dim_VendorTechnology.csv

- Dim_Site.csv

- Dim_Fault.csv

- Fact_Incident.csv



These files are used for MySQL analysis and Power BI reporting. All data in this project is synthetic and created for portfolio and learning purposes.

## Project Assumptions & Business Rules 

#### Objective Generate a synthetic telecom service assurance dataset that can be used for SQL analysis and Power BI reporting. 

#### Dataset Scope
 - Dataset Period: January 2023 – December 2025
 - Total Incidents: 25,000
 - Sites: 650
 - States / UTs: 18
 - Geography Mix:
     - Urban: 50%
     - Semi-Urban: 30%
     - Rural: 20% 
- Severity Distribution
     - Low: 60%
     - Medium: 25% 
     - High: 10% 
     - Critical: 5% 

#### Generation Logic
 The dataset is built in stages instead of generating every field independently.
 The process starts with the core attributes:
 - Date
 - Site
 - Vendor Technology
 - Fault 
The remaining fields are calculated from these values using predefined business rules.
 Fault 
    ↓
 Severity
    ↓ 
 Resolution Type
    ↓ 
 Resolution Minutes
    ↓
 SLA Breach
    ↓
 Customers Impacted
    ↓
 Dispatch
    ↓
 Estimated Cost
    ↓
 Estimated Revenue Loss
    ↓
 Repeat Fault
    ↓
 Escalation
 This approach keeps the relationships between fields consistent throughout the dataset.

 #### Validation
 Before exporting the data, the notebook checks for:
 - Duplicate Incident IDs
 - Null values
 - Foreign key integrity
 - Referential consistency

 #### Output Files
 The notebook exports the following CSV files:
- Dim_Date.csv
- Dim_VendorTechnology.csv
- Dim_Site.csv
- Dim_Fault.csv
- Fact_Incident.csv

## 1. Setup
Import the required libraries before creating the dimension and fact tables.

In [1]:
import pandas as pd
import numpy as np

# Reproducible synthetic dataset
np.random.seed(42)

## 2. Date Dimension
Create the Date Dimension covering January 2023 to December 2025.
This table will be used throughout the project for date-based analysis.

In [2]:
dates = pd.date_range(
    start='2023-01-01',
    end='2025-12-31',
    freq='D'
)

In [3]:
dim_date = pd.DataFrame({
    "Date": dates
})

dim_date["Date_ID"] = dim_date["Date"].dt.strftime("%Y%m%d").astype(int)
dim_date["Year"] = dim_date["Date"].dt.year
dim_date["Quarter"] = "Q" + dim_date["Date"].dt.quarter.astype(str)
dim_date["Month_Number"] = dim_date["Date"].dt.month
dim_date["Month_Name"] = dim_date["Date"].dt.month_name()
dim_date["Month_Short"] = dim_date["Date"].dt.strftime("%b")
dim_date["Week_Number"] = dim_date["Date"].dt.isocalendar().week.astype(int)
dim_date["Day_Number"] = dim_date["Date"].dt.day
dim_date["Day_Name"] = dim_date["Date"].dt.day_name()
dim_date["Is_Weekend"] = dim_date["Date"].dt.weekday >= 5
dim_date["Year_Month"] = dim_date["Date"].dt.strftime("%Y-%m")
dim_date["Year_Month_Label"] = dim_date["Date"].dt.strftime("%b-%Y")

dim_date

,Date,Date_ID,Year,Quarter,Month_Number,Month_Name,Month_Short,Week_Number,Day_Number,Day_Name,Is_Weekend,Year_Month,Year_Month_Label
0,2023-01-01,20230101,2023,Q1,1,January,Jan,52,1,Sunday,True,2023-01,Jan-2023
1,2023-01-02,20230102,2023,Q1,1,January,Jan,1,2,Monday,False,2023-01,Jan-2023
2,2023-01-03,20230103,2023,Q1,1,January,Jan,1,3,Tuesday,False,2023-01,Jan-2023
3,2023-01-04,20230104,2023,Q1,1,January,Jan,1,4,Wednesday,False,2023-01,Jan-2023
4,2023-01-05,20230105,2023,Q1,1,January,Jan,1,5,Thursday,False,2023-01,Jan-2023
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1091,2025-12-27,20251227,2025,Q4,12,December,Dec,52,27,Saturday,True,2025-12,Dec-2025
1092,2025-12-28,20251228,2025,Q4,12,December,Dec,52,28,Sunday,True,2025-12,Dec-2025
1093,2025-12-29,20251229,2025,Q4,12,December,Dec,1,29,Monday,False,2025-12,Dec-2025
1094,2025-12-30,20251230,2025,Q4,12,December,Dec,1,30,Tuesday,False,2025-12,Dec-2025


In [4]:
dim_date.to_csv(
    'Dim_Date.csv',
    index=False
)

## 3. Vendor Technology Dimension
Vendor MTTR affects resolution time. Vendor SLA Risk and Vendor Reliability are included for performance analysis in SQL and Power BI.

In [5]:
vendor_tech_data = [
    ['Huawei', 'GPON'],
    ['Huawei', 'XGS-PON'],
    ['Huawei', 'BRAS'],
    ['Nokia', 'GPON'],
    ['Nokia', 'XGS-PON'],
    ['Nokia', 'BRAS'],
    ['ZTE', 'GPON'],
    ['ZTE', 'BRAS'],
    ['Tejas', 'GPON'],
    ['Tejas', 'BRAS']
]

In [6]:
dim_vendor_technology = pd.DataFrame(
    vendor_tech_data,
    columns=['Vendor', 'Technology']
)

In [7]:
dim_vendor_technology['VendorTech_ID'] = range(
    1,
    len(dim_vendor_technology) + 1
)

In [8]:
dim_vendor_technology["Vendor_MTTR_Profile"] = [
    0.95,
    0.97,
    0.98,
    1.00,
    1.02,
    1.03,
    1.05,
    1.06,
    1.08,
    1.10
]

In [9]:
dim_vendor_technology['Vendor_SLA_Risk_Profile'] = [
    0.05,
    0.06,
    0.04,
    0.07,
    0.08,
    0.07,
    0.09,
    0.10,
    0.12,
    0.14
]

In [10]:
dim_vendor_technology['Vendor_Reliability_Profile'] = [
    0.03,
    0.04,
    0.03,
    0.05,
    0.05,
    0.05,
    0.07,
    0.08,
    0.10,
    0.12
]

### Note
Vendor profile values are included for analysis and reporting. Incident generation is based on fault characteristics.

In [11]:
dim_vendor_technology

,Vendor,Technology,VendorTech_ID,Vendor_MTTR_Profile,Vendor_SLA_Risk_Profile,Vendor_Reliability_Profile
0,Huawei,GPON,1,0.95,0.05,0.03
1,Huawei,XGS-PON,2,0.97,0.06,0.04
2,Huawei,BRAS,3,0.98,0.04,0.03
3,Nokia,GPON,4,1.00,0.07,0.05
4,Nokia,XGS-PON,5,1.02,0.08,0.05
5,Nokia,BRAS,6,1.03,0.07,0.05
6,ZTE,GPON,7,1.05,0.09,0.07
7,ZTE,BRAS,8,1.06,0.10,0.08
8,Tejas,GPON,9,1.08,0.12,0.10
9,Tejas,BRAS,10,1.10,0.14,0.12


In [12]:
dim_vendor_technology.to_csv(
    'Dim_VendorTechnology.csv',
    index=False
)

## 4. Site Dimension
Create the site master used throughout the project.
This table stores the location and operational details for each telecom site.

In [13]:
site_ids = [
    f'SITE{i:04d}'
    for i in range(1, 651)
]

In [14]:
zones = [
    'North',
    'West',
    'Central',
    'East'
]

In [15]:
site_types = np.random.choice(
    ['Urban', 'Semi-Urban', 'Rural'],
    size=650,
    p=[0.50, 0.30, 0.20]
)

In [16]:
site_criticality = []

for site_type in site_types:

    if site_type == 'Urban':
        criticality = np.random.choice(
            ['Tier 1', 'Tier 2', 'Tier 3'],
            p=[0.30, 0.50, 0.20]
        )

    elif site_type == 'Semi-Urban':
        criticality = np.random.choice(
            ['Tier 1', 'Tier 2', 'Tier 3'],
            p=[0.10, 0.50, 0.40]
        )

    else:
        criticality = np.random.choice(
            ['Tier 1', 'Tier 2', 'Tier 3'],
            p=[0.02, 0.18, 0.80]
        )

    site_criticality.append(criticality)

In [17]:
power_backup = []

for site_type in site_types:

    if site_type == 'Urban':
        backup = np.random.choice(
            ['Grid', 'Hybrid', 'DG'],
            p=[0.60, 0.30, 0.10]
        )

    elif site_type == 'Semi-Urban':
        backup = np.random.choice(
            ['Grid', 'DG', 'Hybrid'],
            p=[0.40, 0.45, 0.15]
        )

    else:
        backup = np.random.choice(
            ['DG', 'Solar', 'Grid'],
            p=[0.60, 0.25, 0.15]
        )

    power_backup.append(backup)

In [18]:
customer_base = []

for site_type, criticality in zip(site_types, site_criticality):

    if site_type == 'Urban':

        if criticality == 'Tier 1':
            customers = np.random.randint(8000, 15001)

        elif criticality == 'Tier 2':
            customers = np.random.randint(5000, 8001)

        else:
            customers = np.random.randint(2500, 5001)

    elif site_type == 'Semi-Urban':

        if criticality == 'Tier 1':
            customers = np.random.randint(4000, 7001)

        elif criticality == 'Tier 2':
            customers = np.random.randint(2000, 4001)

        else:
            customers = np.random.randint(1000, 2001)

    else:

        if criticality == 'Tier 1':
            customers = np.random.randint(1500, 3001)

        elif criticality == 'Tier 2':
            customers = np.random.randint(700, 1501)

        else:
            customers = np.random.randint(200, 701)

    customer_base.append(customers)

In [19]:
state_zone_map = {
    'Punjab': 'North',
    'Haryana': 'North',
    'Delhi': 'North',
    'Chandigarh': 'North',
    'Jammu & Kashmir': 'North',
    'Himachal Pradesh': 'North',
    'Uttarakhand': 'North',

    'Maharashtra': 'West',
    'Gujarat': 'West',
    'Rajasthan': 'West',
    'Goa': 'West',

    'Uttar Pradesh': 'Central',
    'Madhya Pradesh': 'Central',
    'Chhattisgarh': 'Central',

    'West Bengal': 'East',
    'Odisha': 'East',
    'Bihar': 'East',
    'Jharkhand': 'East'
}

In [20]:
state_weights = {
    'Punjab': 0.07,
    'Haryana': 0.09,
    'Delhi': 0.10,
    'Chandigarh': 0.025,
    'Jammu & Kashmir': 0.035,
    'Himachal Pradesh': 0.02,
    'Uttarakhand': 0.04,

    'Maharashtra': 0.12,
    'Gujarat': 0.08,
    'Rajasthan': 0.07,
    'Goa': 0.01,

    'Uttar Pradesh': 0.14,
    'Madhya Pradesh': 0.045,
    'Chhattisgarh': 0.02,

    'West Bengal': 0.055,
    'Odisha': 0.03,
    'Bihar': 0.04,
    'Jharkhand': 0.01
}

In [21]:
site_states = np.random.choice(
    list(state_weights.keys()),
    size=650,
    p=list(state_weights.values())
)

In [22]:
site_zones = [
    state_zone_map[state]
    for state in site_states
]

In [23]:
state_circle_map = {

    'Punjab': 'Upper North',
    'Haryana': 'Upper North',
    'Chandigarh': 'Upper North',
    'Jammu & Kashmir': 'Upper North',
    'Himachal Pradesh': 'Upper North',

    'Delhi': 'Delhi NCR',

    'Uttar Pradesh': 'UP',
    'Uttarakhand': 'UP',

    'Maharashtra': 'Maharashtra',
    'Goa': 'Maharashtra',

    'Gujarat': 'Gujarat',

    'Rajasthan': 'Rajasthan',

    'Madhya Pradesh': 'Central',
    'Chhattisgarh': 'Central',

    'Bihar': 'Bihar',
    'Jharkhand': 'Bihar',

    'Odisha': 'East',
    'West Bengal': 'East'
}

In [24]:
site_circles = [
    state_circle_map[state]
    for state in site_states
]

In [25]:
city_map = {

    'Punjab': [
        'Ludhiana','Amritsar','Jalandhar','Patiala','Bathinda'
    ],

    'Haryana': [
        'Gurugram','Faridabad','Panipat','Karnal','Hisar'
    ],

    'Delhi': [
        'New Delhi','Dwarka','Rohini','Saket','Karol Bagh'
    ],

    'Chandigarh': [
        'Chandigarh'
    ],

    'Jammu & Kashmir': [
        'Jammu','Srinagar'
    ],

    'Himachal Pradesh': [
        'Shimla','Dharamshala','Solan'
    ],

    'Uttarakhand': [
        'Dehradun','Haridwar','Haldwani'
    ],

    'Uttar Pradesh': [
        'Lucknow','Noida','Ghaziabad','Kanpur','Varanasi'
    ],

    'Rajasthan': [
        'Jaipur','Jodhpur','Udaipur','Kota'
    ],

    'Maharashtra': [
        'Mumbai','Pune','Nagpur','Nashik'
    ],

    'Goa': [
        'Panaji','Margao'
    ],

    'Gujarat': [
        'Ahmedabad','Surat','Vadodara','Rajkot'
    ],

    'Madhya Pradesh': [
        'Bhopal','Indore','Gwalior'
    ],

    'Chhattisgarh': [
        'Raipur','Bilaspur'
    ],

    'Bihar': [
        'Patna','Gaya','Muzaffarpur'
    ],

    'Jharkhand': [
        'Ranchi','Jamshedpur','Dhanbad'
    ],

    'Odisha': [
        'Bhubaneswar','Cuttack','Rourkela'
    ],

    'West Bengal': [
        'Kolkata','Howrah','Siliguri','Durgapur'
    ]
}

In [26]:
site_cities = [
    np.random.choice(city_map[state])
    for state in site_states
]

In [27]:
site_clusters = [
    f"{city} Cluster"
    for city in site_cities
]

In [28]:
cluster_map = {

    # Punjab
   'Ludhiana': 'Ludhiana',
   'Jalandhar': 'Ludhiana',
   'Patiala': 'Ludhiana',
   'Bathinda': 'Ludhiana',
   'Chandigarh': 'Ludhiana',
   'Amritsar': 'Amritsar',

    # Haryana + NCR
    'Gurugram': 'Delhi NCR',
    'Faridabad': 'Delhi NCR',
    'Panipat': 'Panipat',
    'Karnal': 'Panipat',
    'Hisar': 'Hisar',

    # Delhi
    'Delhi': 'Delhi NCR',
    'New Delhi': 'Delhi NCR',
    'Dwarka': 'Delhi NCR',
    'Rohini': 'Delhi NCR',
    'Saket': 'Delhi NCR',
    'Karol Bagh': 'Delhi NCR',

    # Jammu & Kashmir
    'Jammu': 'Jammu',
    'Srinagar': 'Srinagar',

    # Himachal Pradesh
    'Shimla': 'Shimla',
    'Dharamshala': 'Shimla',
    'Solan': 'Shimla',

    # Uttarakhand
    'Dehradun': 'Dehradun',
    'Haridwar': 'Dehradun',
    'Haldwani': 'Haldwani',

    # Uttar Pradesh
    'Lucknow': 'Lucknow',
    'Kanpur': 'Lucknow',
    'Varanasi': 'Lucknow',
    'Noida': 'Delhi NCR',
    'Ghaziabad': 'Delhi NCR',

    # Rajasthan
    'Jaipur': 'Jaipur',
    'Jodhpur': 'Jaipur',
    'Udaipur': 'Jaipur',
    'Kota': 'Jaipur',

    # Maharashtra
    'Mumbai': 'Mumbai',
    'Pune': 'Mumbai',
    'Nashik': 'Mumbai',
    'Nagpur': 'Nagpur',

    # Goa
    'Panaji': 'Mumbai',
    'Margao': 'Mumbai',

    # Gujarat
    'Ahmedabad': 'Ahmedabad',
    'Surat': 'Ahmedabad',
    'Vadodara': 'Ahmedabad',
    'Rajkot': 'Ahmedabad',

    # Madhya Pradesh
    'Bhopal': 'Bhopal',
    'Indore': 'Indore',
    'Gwalior': 'Gwalior',

    # Chhattisgarh
    'Raipur': 'Raipur',
    'Bilaspur': 'Raipur',

    # Bihar
    'Patna': 'Patna',
    'Gaya': 'Patna',
    'Muzaffarpur': 'Patna',

    # Jharkhand
    'Ranchi': 'Ranchi',
    'Jamshedpur': 'Ranchi',
    'Dhanbad': 'Ranchi',

    # Odisha
    'Bhubaneswar': 'Bhubaneswar',
    'Cuttack': 'Bhubaneswar',
    'Rourkela': 'Bhubaneswar',

    # West Bengal
    'Kolkata': 'Kolkata',
    'Howrah': 'Kolkata',
    'Siliguri': 'Kolkata',
    'Durgapur': 'Kolkata'
}

In [29]:
site_clusters = [cluster_map[city] for city in site_cities]

In [30]:
vendor_tech_ids = [1,2,3,4,5,6,7,8,9,10]

vendor_weights = [
    0.22,   
    0.03,  
    0.08,  
    0.22, 
    0.04,   
    0.08,   
    0.14,   
    0.07,   
    0.08,   
    0.04    
]

primary_vendor_tech_ids = np.random.choice(
    vendor_tech_ids,
    size=650,
    p=vendor_weights
)

In [31]:
pd.Series(primary_vendor_tech_ids).value_counts().sort_index()

1     149
2      23
3      54
4     130
5      33
6      47
7      96
8      38
9      60
10     20
Name: count, dtype: int64

In [32]:
sum(state_weights.values())

1.0

In [33]:
dim_site = pd.DataFrame({
    'Site_ID': site_ids,
    'Zone': site_zones,
    'State_UT': site_states,
    'Circle': site_circles,
    'City': site_cities,
    'Site_Cluster': site_clusters,
    'Site_Type': site_types,
    'Site_Criticality': site_criticality,
    'Customer_Base': customer_base,
    'Power_Backup': power_backup,
    'Primary_VendorTech_ID': primary_vendor_tech_ids
})

In [34]:
dim_site.to_csv(
    "Dim_Site.csv",
    index=False
)

## 5. Fault Dimension
Create the fault master used while generating telecom incidents.
Each fault includes the basic information required for incident generation.

In [35]:
faults = [

# Optical Network

{"Fault_ID":"F001","Fault_Category":"Optical Network","Fault_Name":"Fiber Cut","Network_Layer":"Access","Fault_Type":"Network"},
{"Fault_ID":"F002","Fault_Category":"Optical Network","Fault_Name":"OLT Down","Network_Layer":"Access","Fault_Type":"Network"},
{"Fault_ID":"F003","Fault_Category":"Optical Network","Fault_Name":"OLT Port Failure","Network_Layer":"Access","Fault_Type":"Network"},
{"Fault_ID":"F004","Fault_Category":"Optical Network","Fault_Name":"PON Port Down","Network_Layer":"Access","Fault_Type":"Network"},
{"Fault_ID":"F005","Fault_Category":"Optical Network","Fault_Name":"Optical Power Low","Network_Layer":"Access","Fault_Type":"Network"},
{"Fault_ID":"F006","Fault_Category":"Optical Network","Fault_Name":"Feeder Fiber Fault","Network_Layer":"Access","Fault_Type":"Network"},
{"Fault_ID":"F007","Fault_Category":"Optical Network","Fault_Name":"Distribution Fiber Fault","Network_Layer":"Access","Fault_Type":"Network"},
{"Fault_ID":"F008","Fault_Category":"Optical Network","Fault_Name":"Splitter Failure","Network_Layer":"Access","Fault_Type":"Network"},

# Access Equipment

{"Fault_ID":"F009","Fault_Category":"Access Equipment","Fault_Name":"ONT Offline","Network_Layer":"Access","Fault_Type":"Equipment"},
{"Fault_ID":"F010","Fault_Category":"Access Equipment","Fault_Name":"ONT Authentication Failure","Network_Layer":"Access","Fault_Type":"Equipment"},
{"Fault_ID":"F011","Fault_Category":"Access Equipment","Fault_Name":"ONT LOS","Network_Layer":"Access","Fault_Type":"Equipment"},
{"Fault_ID":"F012","Fault_Category":"Access Equipment","Fault_Name":"ONT Power Failure","Network_Layer":"Access","Fault_Type":"Equipment"},
{"Fault_ID":"F013","Fault_Category":"Access Equipment","Fault_Name":"ONT Registration Failure","Network_Layer":"Access","Fault_Type":"Equipment"},

# Core & IP

{"Fault_ID":"F014","Fault_Category":"Core & IP","Fault_Name":"BRAS Down","Network_Layer":"Core","Fault_Type":"Network"},
{"Fault_ID":"F015","Fault_Category":"Core & IP","Fault_Name":"BNG Authentication Failure","Network_Layer":"Core","Fault_Type":"Network"},
{"Fault_ID":"F016","Fault_Category":"Core & IP","Fault_Name":"DHCP Failure","Network_Layer":"Core","Fault_Type":"Network"},
{"Fault_ID":"F017","Fault_Category":"Core & IP","Fault_Name":"PPPoE Authentication Failure","Network_Layer":"Core","Fault_Type":"Network"},
{"Fault_ID":"F018","Fault_Category":"Core & IP","Fault_Name":"DNS Failure","Network_Layer":"Core","Fault_Type":"Network"},
{"Fault_ID":"F019","Fault_Category":"Core & IP","Fault_Name":"Gateway Reachability Failure","Network_Layer":"Core","Fault_Type":"Network"},

# Transport
    
{"Fault_ID":"F020","Fault_Category":"Transport","Fault_Name":"Uplink Down","Network_Layer":"Transport","Fault_Type":"Network"},
{"Fault_ID":"F021","Fault_Category":"Transport","Fault_Name":"Aggregation Link Failure","Network_Layer":"Transport","Fault_Type":"Network"},
{"Fault_ID":"F022","Fault_Category":"Transport","Fault_Name":"Ring Break","Network_Layer":"Transport","Fault_Type":"Network"},
{"Fault_ID":"F023","Fault_Category":"Transport","Fault_Name":"Backhaul Congestion","Network_Layer":"Transport","Fault_Type":"Network"},

# Customer Premises

{"Fault_ID":"F024","Fault_Category":"Customer Premises","Fault_Name":"Wi-Fi Issue","Network_Layer":"Customer","Fault_Type":"Customer"},
{"Fault_ID":"F025","Fault_Category":"Customer Premises","Fault_Name":"LAN Port Issue","Network_Layer":"Customer","Fault_Type":"Customer"},
{"Fault_ID":"F026","Fault_Category":"Customer Premises","Fault_Name":"Router Configuration Issue","Network_Layer":"Customer","Fault_Type":"Customer"},
{"Fault_ID":"F027","Fault_Category":"Customer Premises","Fault_Name":"Customer Power Off","Network_Layer":"Customer","Fault_Type":"Customer"},
{"Fault_ID":"F028","Fault_Category":"Customer Premises","Fault_Name":"Patch Cord Issue","Network_Layer":"Customer","Fault_Type":"Customer"},
{"Fault_ID":"F029","Fault_Category":"Customer Premises","Fault_Name":"Indoor Fiber Damage","Network_Layer":"Customer","Fault_Type":"Customer"},

# Power & Environment

{"Fault_ID":"F030","Fault_Category":"Power & Environment","Fault_Name":"Power Failure","Network_Layer":"Site","Fault_Type":"Infrastructure"},
{"Fault_ID":"F031","Fault_Category":"Power & Environment","Fault_Name":"Battery Backup Failure","Network_Layer":"Site","Fault_Type":"Infrastructure"},
{"Fault_ID":"F032","Fault_Category":"Power & Environment","Fault_Name":"High Temperature Alarm","Network_Layer":"Site","Fault_Type":"Infrastructure"},

# Configuration

{"Fault_ID":"F033","Fault_Category":"Configuration","Fault_Name":"Software Bug","Network_Layer":"System","Fault_Type":"Configuration"},
{"Fault_ID":"F034","Fault_Category":"Configuration","Fault_Name":"Firmware Upgrade Failure","Network_Layer":"System","Fault_Type":"Configuration"},
{"Fault_ID":"F035","Fault_Category":"Configuration","Fault_Name":"Provisioning Error","Network_Layer":"System","Fault_Type":"Configuration"},
{"Fault_ID":"F036","Fault_Category":"Configuration","Fault_Name":"SDN Controller Synchronization Failure","Network_Layer":"System","Fault_Type":"Configuration"},

# Third Party

{"Fault_ID":"F037","Fault_Category":"Third Party","Fault_Name":"Third-Party Fiber Damage","Network_Layer":"External","Fault_Type":"External"},
{"Fault_ID":"F038","Fault_Category":"Third Party","Fault_Name":"Upstream Provider Outage","Network_Layer":"External","Fault_Type":"External"},

# Vendor Hardware

{"Fault_ID":"F039","Fault_Category":"Vendor Hardware","Fault_Name":"Access Equipment Hardware Failure","Network_Layer":"Access","Fault_Type":"Hardware"},
{"Fault_ID":"F040","Fault_Category":"Vendor Hardware","Fault_Name":"Core Network Hardware Failure","Network_Layer":"Core","Fault_Type":"Hardware"}

]

In [36]:
dim_fault = pd.DataFrame(faults)

In [37]:
print("DIM_FAULT VALIDATION")
print(f"Rows                 : {len(dim_fault)}")
print(f"Columns              : {len(dim_fault.columns)}")

print(f"\nUnique Fault_ID      : {dim_fault['Fault_ID'].is_unique}")
print(f"Unique Fault_Name    : {dim_fault['Fault_Name'].is_unique}")

print("\nNull Values")
print(dim_fault.isnull().sum())

print("\nFault Category Distribution")
print(dim_fault["Fault_Category"].value_counts())

print("\nNetwork Layer Distribution")
print(dim_fault["Network_Layer"].value_counts())

DIM_FAULT VALIDATION
Rows                 : 40
Columns              : 5

Unique Fault_ID      : True
Unique Fault_Name    : True

Null Values
Fault_ID          0
Fault_Category    0
Fault_Name        0
Network_Layer     0
Fault_Type        0
dtype: int64

Fault Category Distribution
Fault_Category
Optical Network        8
Core & IP              6
Customer Premises      6
Access Equipment       5
Transport              4
Configuration          4
Power & Environment    3
Third Party            2
Vendor Hardware        2
Name: count, dtype: int64

Network Layer Distribution
Network_Layer
Access       14
Core          7
Customer      6
Transport     4
System        4
Site          3
External      2
Name: count, dtype: int64


In [38]:
fault_weights = {
    "F001": 10,   # Fiber Cut
    "F002": 3,
    "F003": 3,
    "F004": 4,
    "F005": 6,
    "F006": 7,
    "F007": 8,
    "F008": 4,

    "F009": 12,
    "F010": 5,
    "F011": 9,
    "F012": 4,
    "F013": 5,

    "F014": 2,
    "F015": 2,
    "F016": 3,
    "F017": 4,
    "F018": 2,
    "F019": 1,

    "F020": 4,
    "F021": 3,
    "F022": 2,
    "F023": 3,

    "F024": 15,
    "F025": 8,
    "F026": 6,
    "F027": 5,
    "F028": 4,
    "F029": 5,

    "F030": 6,
    "F031": 3,
    "F032": 2,

    "F033": 2,
    "F034": 2,
    "F035": 4,
    "F036": 2,

    "F037": 2,
    "F038": 1,

    "F039": 2,
    "F040": 1
}

dim_fault["Incident_Share_Weight"] = dim_fault["Fault_ID"].map(fault_weights)

dim_fault.head()

,Fault_ID,Fault_Category,Fault_Name,Network_Layer,Fault_Type,Incident_Share_Weight
0,F001,Optical Network,Fiber Cut,Access,Network,10
1,F002,Optical Network,OLT Down,Access,Network,3
2,F003,Optical Network,OLT Port Failure,Access,Network,3
3,F004,Optical Network,PON Port Down,Access,Network,4
4,F005,Optical Network,Optical Power Low,Access,Network,6


In [39]:
dim_fault.to_csv("Dim_Fault.csv", index=False)

print("Updated Dim_Fault.csv exported successfully.")

Updated Dim_Fault.csv exported successfully.


In [40]:
dim_date = pd.read_csv("Dim_Date.csv")
dim_site = pd.read_csv("Dim_Site.csv")
dim_vendor_technology = pd.read_csv("Dim_VendorTechnology.csv")
dim_fault = pd.read_csv("Dim_Fault.csv")

## 6. Incident Data
Build the incident dataset by combining all dimension tables and applying the business rules defined for this project. Each row represents one broadband service incident.

### 6.1 Initialize Dataset
Load the completed dimension tables and create the base structure for the incident dataset.

In [41]:
TOTAL_INCIDENTS = 25000

fact_incident = pd.DataFrame({
    "Incident_ID": [
        f"INC{str(i).zfill(6)}"
        for i in range(1, TOTAL_INCIDENTS + 1)
    ]
})

print(fact_incident.head())

print("\nTotal Incidents:", len(fact_incident))

  Incident_ID
0   INC000001
1   INC000002
2   INC000003
3   INC000004
4   INC000005

Total Incidents: 25000


### 6.2 Assign Root Attributes
Assign the core attributes required for every incident before generating the remaining business fields.

In [42]:
date_2023 = dim_date.loc[dim_date["Year"] == 2023, "Date_ID"]
date_2024 = dim_date.loc[dim_date["Year"] == 2024, "Date_ID"]
date_2025 = dim_date.loc[dim_date["Year"] == 2025, "Date_ID"]

date_ids = np.concatenate([

    np.random.choice(date_2023, 6250, replace=True),

    np.random.choice(date_2024, 7900, replace=True),

    np.random.choice(date_2025, 10850, replace=True)

])

np.random.shuffle(date_ids)

fact_incident["Date_ID"] = date_ids

print(fact_incident.head())

  Incident_ID   Date_ID
0   INC000001  20230617
1   INC000002  20250828
2   INC000003  20240630
3   INC000004  20230419
4   INC000005  20250809


In [43]:
site_type_weight = {
    "Urban": 1.30,
    "Semi-Urban": 1.00,
    "Rural": 0.70
}

# Site Criticality Weight

criticality_weight = {
    "Tier 1": 1.40,
    "Tier 2": 1.00,
    "Tier 3": 0.70
}

# Normalize Customer Base
customer_weight = dim_site["Customer_Base"] / dim_site["Customer_Base"].mean()

# Final Sampling Weight
dim_site["Sampling_Weight"] = (
    customer_weight
    * dim_site["Site_Type"].map(site_type_weight)
    * dim_site["Site_Criticality"].map(criticality_weight)
)

# Convert to probabilities
dim_site["Sampling_Probability"] = (
    dim_site["Sampling_Weight"] /
    dim_site["Sampling_Weight"].sum()
)

dim_site[[
    "Site_Type",
    "Site_Criticality",
    "Customer_Base",
    "Sampling_Probability"
]].head()

,Site_Type,Site_Criticality,Customer_Base,Sampling_Probability
0,Urban,Tier 1,9169,0.003815
1,Rural,Tier 3,518,0.000058
2,Semi-Urban,Tier 2,3651,0.000835
3,Semi-Urban,Tier 3,1112,0.000178
4,Urban,Tier 2,7810,0.002321


In [44]:
# Normalize probabilities (safety check)

site_p = (
    dim_site["Sampling_Probability"] /
    dim_site["Sampling_Probability"].sum()
)

fact_incident["Site_ID"] = np.random.choice(
    dim_site["Site_ID"],
    size=TOTAL_INCIDENTS,
    p=site_p
)

In [45]:
# Inherit vendor from selected site

site_vendor = dim_site[["Site_ID", "Primary_VendorTech_ID"]]

fact_incident = fact_incident.merge(
    site_vendor,
    on="Site_ID",
    how="left"
)

fact_incident.rename(
    columns={"Primary_VendorTech_ID": "VendorTech_ID"},
    inplace=True
)

fact_incident.head()

,Incident_ID,Date_ID,Site_ID,VendorTech_ID
0,INC000001,20230617,SITE0295,8
1,INC000002,20250828,SITE0210,3
2,INC000003,20240630,SITE0360,4
3,INC000004,20230419,SITE0469,9
4,INC000005,20250809,SITE0455,6


In [46]:
# Normalize fault probabilities

fault_p = (
    dim_fault["Incident_Share_Weight"] /
    dim_fault["Incident_Share_Weight"].sum()
)

fact_incident["Fault_ID"] = np.random.choice(
    dim_fault["Fault_ID"],
    size=TOTAL_INCIDENTS,
    p=fault_p
)

In [47]:
fact_incident = fact_incident.merge(
    dim_fault[
        [
            "Fault_ID",
            "Fault_Name",
            "Fault_Category",
            "Network_Layer",
            "Fault_Type"
        ]
    ],
    on="Fault_ID",
    how="left"
)

In [48]:
fact_incident = fact_incident.merge(
    dim_vendor_technology[
        ["VendorTech_ID", "Vendor_MTTR_Profile"]
    ],
    on="VendorTech_ID",
    how="left"
)

### 6.3 Generate Incident Severity
Assign severity based on the selected fault category using the predefined severity matrix.

In [49]:
severity_matrix = {

    "Customer Premises":     [0.85, 0.14, 0.01, 0.00],

    "Access Equipment":      [0.70, 0.25, 0.05, 0.00],

    "Optical Network":       [0.55, 0.30, 0.12, 0.03],

    "Core & IP":             [0.35, 0.35, 0.20, 0.10],

    "Transport":             [0.40, 0.35, 0.20, 0.05],

    "Power & Environment":   [0.50, 0.30, 0.15, 0.05],

    "Configuration":         [0.60, 0.25, 0.10, 0.05],

    "Third Party":           [0.40, 0.30, 0.20, 0.10],

    "Vendor Hardware":       [0.40, 0.30, 0.20, 0.10]

}

In [50]:
severity_levels = ["Low", "Medium", "High", "Critical"]

fact_incident["Severity"] = fact_incident["Fault_Category"].apply(
    lambda category: np.random.choice(
        severity_levels,
        p=severity_matrix[category]
    )
)

### 6.4 Resolution Details
Generate the resolution approach and calculate the expected resolution time for each incident based on fault category and severity.

In [51]:
# Probabilistic Resolution Type Mapping

resolution_type_matrix = {

    "Customer Premises": (
        ["Remote Resolution", "Field Visit"],
        [0.95, 0.05]
    ),

    "Access Equipment": (
        ["Field Visit", "Remote Configuration"],
        [0.90, 0.10]
    ),

    "Optical Network": (
        ["Fiber Team Dispatch", "Vendor Support", "Hardware Replacement"],
        [0.85, 0.10, 0.05]
    ),

    "Core & IP": (
        ["NOC Support", "Vendor Support"],
        [0.90, 0.10]
    ),

    "Transport": (
        ["NOC Support", "Vendor Support"],
        [0.85, 0.15]
    ),

    "Power & Environment": (
        ["Field Visit", "Vendor Support"],
        [0.90, 0.10]
    ),

    "Configuration": (
        ["Remote Configuration", "NOC Support"],
        [0.95, 0.05]
    ),

    "Third Party": (
        ["Vendor Support", "Fiber Team Dispatch"],
        [0.90, 0.10]
    ),

    "Vendor Hardware": (
        ["Hardware Replacement", "Vendor Support"],
        [0.90, 0.10]
    )

}


fact_incident["Resolution_Type"] = (
    fact_incident["Fault_Category"]
    .apply(
        lambda x: np.random.choice(
            resolution_type_matrix[x][0],
            p=resolution_type_matrix[x][1]
        )
    )
)

In [52]:
resolution_time_ranges = {

    "Remote Resolution": (20, 60),

    "Remote Configuration": (30, 90),

    "NOC Support": (60, 180),

    "Field Visit": (120, 360),

    "Fiber Team Dispatch": (240, 600),

    "Vendor Support": (360, 720),

    "Hardware Replacement": (480, 960)

}

# Generate base resolution time from Resolution Type
fact_incident["Resolution_Minutes"] = (
    fact_incident["Resolution_Type"]
    .apply(
        lambda x: np.random.randint(
            resolution_time_ranges[x][0],
            resolution_time_ranges[x][1] + 1
        )
    )
)

# Severity adjustment
severity_multiplier = {
    "Low": 1.10,
    "Medium": 1.00,
    "High": 0.90,
    "Critical": 0.80
}

fact_incident["Resolution_Minutes"] = (
    fact_incident["Resolution_Minutes"]
    * fact_incident["Severity"].map(severity_multiplier)
    * fact_incident["Vendor_MTTR_Profile"]
).round().astype(int)

### 6.5 SLA Evaluation
Assign SLA targets and identify SLA breaches.

In [53]:
sla_target_map = {
    "Remote Resolution": {
        "Low": 90,
        "Medium": 75,
        "High": 70,
        "Critical": 60
    },

    "Remote Configuration": {
        "Low": 120,
        "Medium": 95,
        "High": 90,
        "Critical": 75
    },

    "NOC Support": {
        "Low": 240,
        "Medium": 190,
        "High": 180,
        "Critical": 150
    },

    "Field Visit": {
        "Low": 360,
        "Medium": 300,
        "High": 270,
        "Critical": 210
    },

    "Fiber Team Dispatch": {
        "Low": 600,
        "Medium": 500,
        "High": 420,
        "Critical": 330
    },

    "Vendor Support": {
        "Low": 900,
        "Medium": 760,
        "High": 660,
        "Critical": 540
    },

    "Hardware Replacement": {
        "Low": 1200,
        "Medium": 1000,
        "High": 840,
        "Critical": 660
    }
}
fact_incident["SLA_Target_Minutes"] = [
    sla_target_map[rt][sev]
    for rt, sev in zip(
        fact_incident["Resolution_Type"],
        fact_incident["Severity"]
    )
]

fact_incident["SLA_Target_Minutes"] = (
    fact_incident["SLA_Target_Minutes"]
).astype(int)

In [54]:
fact_incident["SLA_Breach"] = np.where(
    fact_incident["Resolution_Minutes"] >
    fact_incident["SLA_Target_Minutes"],
    "Yes",
    "No"
)

fact_incident["SLA_Breach"].value_counts(normalize=True) * 100

SLA_Breach
No     90.012
Yes     9.988
Name: proportion, dtype: float64

In [55]:
(
    fact_incident
    .groupby("Severity")["SLA_Breach"]
    .apply(lambda x: (x == "Yes").mean() * 100)
    .round(2)
)

Severity
Critical    19.65
High        15.36
Low          7.12
Medium      14.07
Name: SLA_Breach, dtype: float64

### 6.6 Customer Impact
Estimate customers affected based on site profile and fault characteristics.

In [56]:
fact_incident = fact_incident.merge(
    dim_site[
        ["Site_ID", "Customer_Base", "Site_Criticality"]
    ],
    on="Site_ID",
    how="left"
)

In [57]:
impact_percentage = {

    "Customer Premises": (0.001, 0.005),

    "Access Equipment": (0.01, 0.05),

    "Optical Network": (0.05, 0.20),

    "Core & IP": (0.20, 0.60),

    "Transport": (0.10, 0.40),

    "Power & Environment": (0.05, 0.15),

    "Configuration": (0.02, 0.08),

    "Third Party": (0.05, 0.20),

    "Vendor Hardware": (0.05, 0.25)

}

In [58]:
criticality_modifier = {

    "Tier 1": 1.20,

    "Tier 2": 1.00,

    "Tier 3": 0.80

}

In [59]:
def generate_customers_impacted(row):

    low, high = impact_percentage[row["Fault_Category"]]

    span = high - low

    r = np.random.random()

    # 70% of incidents are small
    if r < 0.70:
        pct = np.random.uniform(low, low + span * 0.33)

    # 20% are medium
    elif r < 0.90:
        pct = np.random.uniform(
            low + span * 0.33,
            low + span * 0.66
        )

    # 10% are large
    else:
        pct = np.random.uniform(
            low + span * 0.66,
            high
        )

    pct *= criticality_modifier[row["Site_Criticality"]]

    customers = int(row["Customer_Base"] * pct)

    return min(customers, row["Customer_Base"])

In [60]:
fact_incident["Customers_Impacted"] = (
    fact_incident.apply(
        generate_customers_impacted,
        axis=1
    )
)

### 6.7 Field Dispatch
Determine whether a field visit is required and estimate dispatch cost.

In [61]:
def generate_dispatch(resolution_type):

    if resolution_type == "Remote Resolution":
        return "No"

    elif resolution_type == "Remote Configuration":
        return np.random.choice(
            ["No", "Yes"],
            p=[0.95, 0.05]
        )

    elif resolution_type == "NOC Support":
        return np.random.choice(
            ["No", "Yes"],
            p=[0.97, 0.03]
        )

    elif resolution_type == "Vendor Support":
        return np.random.choice(
            ["No", "Yes"],
            p=[0.70, 0.30]
        )

    elif resolution_type in [
        "Field Visit",
        "Fiber Team Dispatch",
        "Hardware Replacement"
    ]:
        return "Yes"

fact_incident["Dispatch_Required"] = (
    fact_incident["Resolution_Type"]
    .apply(generate_dispatch)
)

In [62]:
fact_incident = fact_incident.merge(
    dim_site[
        ["Site_ID", "Site_Type"]
    ],
    on="Site_ID",
    how="left"
)

### 6.8 Dispatch Cost
Estimate dispatch expenses based on the selected resolution approach and site characteristics.

In [63]:
site_multiplier = {
    "Urban": 1.00,
    "Semi-Urban": 1.10,
    "Rural": 1.25
}

def generate_dispatch_cost(row):

    # No dispatch = No cost
    if row["Dispatch_Required"] == "No":
        return 0
         
    # Base cost by resolution type    
    if row["Resolution_Type"] == "Field Visit":
        base_cost = np.random.randint(800, 1501)

    elif row["Resolution_Type"] == "Fiber Team Dispatch":
        base_cost = np.random.randint(2500, 5001)

    elif row["Resolution_Type"] == "Vendor Support":
        base_cost = np.random.randint(1500, 3001)

    elif row["Resolution_Type"] == "NOC Support":
        base_cost = np.random.randint(250, 501)

    elif row["Resolution_Type"] == "Remote Configuration":
        base_cost = np.random.randint(150, 301)

    elif row["Resolution_Type"] == "Hardware Replacement":
        base_cost = np.random.randint(5000, 12001)

    else:
        base_cost = 0

    # Apply Site Type multiplier
    multiplier = site_multiplier.get(
        row["Site_Type"],
        1.00
    )

    final_cost = int(base_cost * multiplier)

    return final_cost


fact_incident["Dispatch_Cost"] = fact_incident.apply(
    generate_dispatch_cost,
    axis=1
)

print("Dispatch Cost generated successfully.")

fact_incident.drop(
    columns=["Site_Type"],
    inplace=True
)

print("Temporary Site_Type column removed.")

Dispatch Cost generated successfully.
Temporary Site_Type column removed.


### 6.9 Operational Resolution Cost

Estimate the internal engineering effort and external vendor support cost required to resolve each incident.

In [64]:
# Internal engineering effort cost
labour_cost = {
    "Remote Resolution": 15,
    "Remote Configuration": 20,
    "NOC Support": 35,
    "Field Visit": 50,
    "Fiber Team Dispatch": 65,
    "Vendor Support": 80,
    "Hardware Replacement": 100
}

# Additional external vendor support cost
vendor_cost = {
    "Remote Resolution": 0,
    "Remote Configuration": 0,
    "NOC Support": 0,
    "Field Visit": 0,
    "Fiber Team Dispatch": 0,
    "Vendor Support": 40,
    "Hardware Replacement": 80
}

fact_incident["Estimated_Operational_Cost"] = (
    fact_incident["Resolution_Type"].map(labour_cost)
    + fact_incident["Dispatch_Cost"]
    + fact_incident["Resolution_Type"].map(vendor_cost)
)

print("Estimated Operational Cost generated successfully.")

Estimated Operational Cost generated successfully.


### 6.10 Service Impact Cost
Estimate the business impact of customer downtime using affected customers, resolution duration, and site criticality.

In [65]:
REVENUE_LOSS_RATE = 0.45  # ₹ per impacted customer per minute

criticality_multiplier = {
    "Tier 1": 1.20,
    "Tier 2": 1.10,
    "Tier 3": 1.00
}

fact_incident["Estimated_Service_Impact_Cost"] = (

    fact_incident["Customers_Impacted"]

    *

    fact_incident["Resolution_Minutes"]

    *

    REVENUE_LOSS_RATE

    *

    fact_incident["Site_Criticality"].map(criticality_multiplier)

).round(2)

### 6.11 Total Incident Cost
Combine operational resolution cost and service impact cost into a single estimated incident cost.

In [66]:
fact_incident["Estimated_Total_Incident_Cost"] = (
    fact_incident["Estimated_Operational_Cost"]
    + fact_incident["Estimated_Service_Impact_Cost"]
).round(2)

print("Estimated_Total_Incident_Cost generated successfully.")

Estimated_Total_Incident_Cost generated successfully.


### 6.12 Repeat Fault Analysis
Repeat faults are identified from incident history at the same site, while vendor reliability is used for vendor performance analysis.

In [67]:
fact_incident["Incident_Date"] = pd.to_datetime(
    fact_incident["Date_ID"].astype(str),
    format="%Y%m%d"
)

In [68]:
fact_incident = fact_incident.sort_values(
    by=[
        "Site_ID",
        "Fault_ID",
        "Incident_Date"
    ]
)

In [69]:
fact_incident["Previous_Incident_Date"] = (

    fact_incident
    .groupby(["Site_ID", "Fault_ID"])["Incident_Date"]
    .shift(1)

)

In [70]:
fact_incident["Days_Since_Last_Fault"] = (

    fact_incident["Incident_Date"]

    -

    fact_incident["Previous_Incident_Date"]

).dt.days

In [71]:
fact_incident["Repeat_Fault"] = np.where(

    fact_incident["Days_Since_Last_Fault"] <= 21,

    "Yes",

    "No"

)

### 6.13 Escalation
Apply escalation rules based on repeat faults, hardware replacement, and SLA breaches.

In [72]:
def generate_escalation(row):

    # Repeat faults always escalate
    if row["Repeat_Fault"] == "Yes":
        return "Yes"

    # Hardware replacements always escalate
    if row["Resolution_Type"] == "Hardware Replacement":
        return "Yes"

    # Most SLA breaches escalate
    if row["SLA_Breach"] == "Yes":
        return np.random.choice(
            ["Yes", "No"],
            p=[0.90, 0.10]
        )

    return "No"


fact_incident["Escalation"] = fact_incident.apply(
    generate_escalation,
    axis=1
)

In [73]:
fact_incident.drop(
    columns=[
        "Incident_Date",
        "Previous_Incident_Date",
        "Days_Since_Last_Fault",
        "Customer_Base",
        "Site_Criticality",
        "Site_Type"
    ],
    inplace=True,
    errors="ignore"
)

## 7. Final Validation
Run integrity checks to confirm the generated dataset is complete and ready for SQL and Power BI.

In [74]:
print("=" * 58)
print("FACT INCIDENT DATA QUALITY REPORT")
print("=" * 58)

print(f"{'Rows':<25}: {len(fact_incident):,}")
print(f"{'Columns':<25}: {fact_incident.shape[1]}")
print(f"{'Duplicate Incident IDs':<25}: {fact_incident['Incident_ID'].duplicated().sum()}")
print(f"{'Total Null Values':<25}: {fact_incident.isnull().sum().sum()}")

print("\nForeign Key Validation")
print("-" * 58)

print(f"{'Date_ID':<25}: {'PASS' if fact_incident['Date_ID'].isin(dim_date['Date_ID']).all() else 'FAIL'}")
print(f"{'Site_ID':<25}: {'PASS' if fact_incident['Site_ID'].isin(dim_site['Site_ID']).all() else 'FAIL'}")
print(f"{'VendorTech_ID':<25}: {'PASS' if fact_incident['VendorTech_ID'].isin(dim_vendor_technology['VendorTech_ID']).all() else 'FAIL'}")
print(f"{'Fault_ID':<25}: {'PASS' if fact_incident['Fault_ID'].isin(dim_fault['Fault_ID']).all() else 'FAIL'}")

print("\n" + "=" * 58)
print("STATUS".ljust(25) + ": DATASET VALIDATED")
print("=" * 58)

FACT INCIDENT DATA QUALITY REPORT
Rows                     : 25,000
Columns                  : 23
Duplicate Incident IDs   : 0
Total Null Values        : 0

Foreign Key Validation
----------------------------------------------------------
Date_ID                  : PASS
Site_ID                  : PASS
VendorTech_ID            : PASS
Fault_ID                 : PASS

STATUS                   : DATASET VALIDATED


In [75]:
fact_incident = (
    fact_incident
    .sort_values("Incident_ID")
    .reset_index(drop=True)
)

## 8. Export
Export the completed dimension tables and incident dataset for the next stages of the project.

In [76]:
dim_date.to_csv("Dim_Date.csv", index=False)

dim_site.to_csv("Dim_Site.csv", index=False)

dim_vendor_technology.to_csv("Dim_VendorTechnology.csv", index=False)

dim_fault.to_csv("Dim_Fault.csv", index=False)

fact_incident.to_csv("Fact_Incident.csv", index=False)